In [ ]:
# Spatial Domain VGG19 Classifier with Comprehensive Metrics
# Uses original VGG19 architecture on RGB images

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os
import warnings
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, precision_score, 
    recall_score, f1_score, confusion_matrix, classification_report
)
import gc

warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""
    
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) 
                              if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((os.path.join(class_dir, img_name), 
                                           self.class_to_idx[class_name]))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label

def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""
    
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dir = os.path.join(data_root, 'Training')
    test_dir = os.path.join(data_root, 'Test')
    
    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset = FruitsDataset(test_dir, transform=transform_test)
    
    return trainset, testset, trainset.classes

# ================== Step 2: Original VGG19 Model ==================

class VGG19Classifier(nn.Module):
    """
    Original VGG19-based classifier for spatial domain images
    """
    
    def __init__(self, num_classes, dropout_rate=0.5):
        super(VGG19Classifier, self).__init__()
        
        # Load pretrained VGG19
        vgg19 = models.vgg19(pretrained=True)
        
        # Store the feature extractor (all conv layers)
        self.features = vgg19.features
        
        # Global Average Pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # The last conv layer in VGG19 outputs 512 channels
        num_features = 512
        
        # Enhanced classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )
        
        self._initialize_new_weights()
    
    def _initialize_new_weights(self):
        """Initialize new layers with proper weights"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Forward through VGG19 feature extractor
        x = self.features(x)
        
        # Global Average Pooling
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        
        # Classifier
        x = self.classifier(x)
        
        return x
    
    def get_activations(self, x):
        """Extract feature maps from last conv layer"""
        x = torch.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)
        x = self.features(x)
        return x

# ================== Step 3: Training Functions ==================

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None
        
    def __call__(self, val_accuracy, model):
        score = val_accuracy
        
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the VGG19 model"""
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if 'classifier' in name:
            new_params.append(param)
        else:
            pretrained_params.append(param)
    
    optimizer = torch.optim.AdamW([
        {'params': pretrained_params, 'lr': lr * 0.01},
        {'params': new_params, 'lr': lr * 0.5}
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )
    
    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)
    
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    best_val_accuracy = 0.0
    best_model_state = None
    
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (images, labels) in enumerate(train_pbar):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            if torch.isnan(images).any() or torch.isinf(images).any():
                print(f"Warning: NaN/Inf detected in input batch {i}, skipping...")
                continue
            
            optimizer.zero_grad(set_to_none=True)
            
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss detected in batch {i}, skipping...")
                    continue
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })
            
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = running_loss / max(len(train_loader), 1)
        train_accuracy = 100 * correct_train / max(total_train, 1)
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)
        
        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for images, labels in val_pbar:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                
                if scaler is not None:
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct / total:.2f}%'
                })
        
        avg_val_loss = running_val_loss / max(len(val_loader), 1)
        val_accuracy = 100 * correct / max(total, 1)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {optimizer.param_groups[0]["lr"]:.6f}')
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        
        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with validation accuracy: {best_val_accuracy:.2f}%")
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 4: Metrics Calculation ==================

def calculate_metrics(y_true, y_pred, num_classes):
    """
    Calculate comprehensive classification metrics
    
    Returns:
        dict: Dictionary containing all metrics
    """
    # Overall Accuracy
    accuracy = accuracy_score(y_true, y_pred)
    
    # Cohen's Kappa Score
    kappa = cohen_kappa_score(y_true, y_pred)
    
    # Overall Precision (macro-averaged)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Overall Recall (macro-averaged)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Overall F1 Score (macro-averaged)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Confusion Matrix for specificity calculation
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate specificity for each class and average
    specificities = []
    for i in range(num_classes):
        # True negatives: all samples not in class i and not predicted as class i
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        # False positives: samples not in class i but predicted as class i
        fp = np.sum(cm[:, i]) - cm[i, i]
        
        if (tn + fp) > 0:
            specificity_i = tn / (tn + fp)
        else:
            specificity_i = 0.0
        specificities.append(specificity_i)
    
    # Overall Specificity (macro-averaged)
    specificity = np.mean(specificities)
    
    # Error Rate
    error_rate = 1.0 - accuracy
    
    metrics = {
        'accuracy': accuracy,
        'kappa': kappa,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'specificity': specificity,
        'error_rate': error_rate,
        'confusion_matrix': cm
    }
    
    return metrics

def print_metrics(metrics, classes):
    """Print metrics in a formatted way"""
    print("\n" + "="*70)
    print("                    OVERALL MODEL PERFORMANCE METRICS")
    print("="*70)
    
    print(f"\n{'Metric':<30} {'Value':>15}")
    print("-"*50)
    print(f"{'Overall Accuracy':<30} {metrics['accuracy']*100:>14.4f}%")
    print(f"{'Overall Error Rate':<30} {metrics['error_rate']*100:>14.4f}%")
    print(f"{'Cohen\\'s Kappa Score':<30} {metrics['kappa']:>15.4f}")
    print(f"{'Overall Precision':<30} {metrics['precision']*100:>14.4f}%")
    print(f"{'Overall Recall':<30} {metrics['recall']*100:>14.4f}%")
    print(f"{'Overall F1 Score':<30} {metrics['f1_score']*100:>14.4f}%")
    print(f"{'Overall Specificity':<30} {metrics['specificity']*100:>14.4f}%")
    print("-"*50)
    
    print("\n" + "="*70)
    print("                    METRIC INTERPRETATIONS")
    print("="*70)
    print(f"""
• Accuracy:      Proportion of correct predictions out of all predictions.
• Error Rate:    Proportion of incorrect predictions (1 - Accuracy).
• Cohen's Kappa: Measures agreement between predictions and true labels,
                 accounting for chance agreement. Values: -1 to 1.
                 (>0.8: excellent, 0.6-0.8: substantial, 0.4-0.6: moderate)
• Precision:     True positives / (True positives + False positives).
                 Of all positive predictions, how many are correct.
• Recall:        True positives / (True positives + False negatives).
                 Of all actual positives, how many were correctly identified.
• F1 Score:      Harmonic mean of Precision and Recall.
• Specificity:   True negatives / (True negatives + False positives).
                 Of all actual negatives, how many were correctly identified.
""")

# ================== Step 5: Visualization Functions ==================

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation curves"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(train_losses) + 1)
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs, val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(cm, classes, max_classes=30):
    """Plot confusion matrix (shows subset if too many classes)"""
    
    if len(classes) > max_classes:
        print(f"\nShowing confusion matrix for first {max_classes} classes (out of {len(classes)})")
        cm_subset = cm[:max_classes, :max_classes]
        classes_subset = classes[:max_classes]
    else:
        cm_subset = cm
        classes_subset = classes
    
    # Normalize confusion matrix
    cm_normalized = cm_subset.astype('float') / cm_subset.sum(axis=1)[:, np.newaxis]
    cm_normalized = np.nan_to_num(cm_normalized)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_normalized, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax)
    
    ax.set(xticks=np.arange(cm_normalized.shape[1]),
           yticks=np.arange(cm_normalized.shape[0]),
           xticklabels=classes_subset, yticklabels=classes_subset,
           title='Normalized Confusion Matrix',
           ylabel='True Label',
           xlabel='Predicted Label')
    
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", fontsize=8)
    plt.setp(ax.get_yticklabels(), fontsize=8)
    
    plt.tight_layout()
    plt.show()

def plot_metrics_bar(metrics):
    """Plot metrics as a bar chart"""
    
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Specificity']
    metric_values = [
        metrics['accuracy'] * 100,
        metrics['precision'] * 100,
        metrics['recall'] * 100,
        metrics['f1_score'] * 100,
        metrics['specificity'] * 100
    ]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12']
    bars = ax.bar(metric_names, metric_values, color=colors, edgecolor='black', linewidth=1.2)
    
    # Add value labels on bars
    for bar, value in zip(bars, metric_values):
        height = bar.get_height()
        ax.annotate(f'{value:.2f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylim(0, 105)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.set_title('Overall Classification Metrics', fontsize=14, fontweight='bold')
    ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ================== Step 6: Main Execution Pipeline ==================

def main():
    print("="*80)
    print("           SPATIAL DOMAIN VGG19 CLASSIFIER FOR FRUITS-360")
    print("              Original VGG19 Architecture with RGB Images")
    print("="*80)
    
    # Dataset path - MODIFY THIS PATH
    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'
    
    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please modify the 'data_root' variable to point to your dataset location.")
        return
    
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        print(f"Number of classes: {len(classes)}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return
    
    print("\n[Step 2] Splitting training set into train/validation...")
    train_size = int(0.85 * len(trainset))
    val_size = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    print(f"Training samples: {len(train_subset)}")
    print(f"Validation samples: {len(val_subset)}")
    print(f"Test samples: {len(testset)}")
    
    batch_size = 64
    num_workers = 4 if os.name != 'nt' else 0
    
    print(f"\nBatch size: {batch_size}")
    print(f"Num workers: {num_workers}")
    
    train_loader = DataLoader(
        train_subset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False if num_workers > 0 else None,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    val_loader = DataLoader(
        val_subset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers, 
        pin_memory=True,
        persistent_workers=False if num_workers > 0 else None,
        prefetch_factor=2 if num_workers > 0 else None
    )
    
    test_loader = DataLoader(
        testset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers
    )
    
    print("\n[Step 3] Initializing VGG19 model...")
    model = VGG19Classifier(num_classes=len(classes), dropout_rate=0.5).to(device)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print("Architecture: VGG19 (pretrained) with custom classifier head")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
        print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    
    print("\n[Step 4] Training model...")
    print("Starting training loop...\n")
    
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader, 
            epochs=50,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        print(f"\nERROR during training: {e}")
        import traceback
        traceback.print_exc()
        return
    
    print("\n[Step 5] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)
    
    print("\n[Step 6] Evaluating on test set...")
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc="Testing")
        for images, labels in test_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    # Calculate comprehensive metrics
    print("\n[Step 7] Calculating comprehensive metrics...")
    metrics = calculate_metrics(all_labels, all_predictions, len(classes))
    
    # Print metrics
    print_metrics(metrics, classes)
    
    # Plot metrics bar chart
    print("\n[Step 8] Plotting metrics visualization...")
    plot_metrics_bar(metrics)
    
    # Plot confusion matrix
    print("\n[Step 9] Plotting confusion matrix...")
    plot_confusion_matrix(metrics['confusion_matrix'], classes)
    
    # Print classification report for top classes
    print("\n[Step 10] Detailed Classification Report (first 20 classes):")
    print("-"*70)
    if len(classes) > 20:
        target_names = classes[:20]
        labels_range = list(range(20))
        print(classification_report(
            [l for l in all_labels if l < 20],
            [p for p, l in zip(all_predictions, all_labels) if l < 20],
            target_names=target_names,
            zero_division=0
        ))
    else:
        print(classification_report(all_labels, all_predictions, 
                                   target_names=classes, zero_division=0))
    
    # Save model
    print("\n[Step 11] Saving trained model...")
    try:
        torch.save({
            'model_state_dict': model.state_dict(),
            'metrics': metrics,
            'classes': classes,
            'num_classes': len(classes)
        }, 'fruits_vgg19_spatial_classifier.pth')
        print("Model saved as 'fruits_vgg19_spatial_classifier.pth'")
    except Exception as e:
        print(f"ERROR saving model: {e}")
    
    # Final summary
    print("\n" + "="*80)
    print("                    FINAL SUMMARY")
    print("="*80)
    print(f"\n{'Metric':<35} {'Value':>20}")
    print("-"*60)
    print(f"{'Total Classes':<35} {len(classes):>20}")
    print(f"{'Training Samples':<35} {len(train_subset):>20}")
    print(f"{'Validation Samples':<35} {len(val_subset):>20}")
    print(f"{'Test Samples':<35} {len(testset):>20}")
    print("-"*60)
    print(f"{'Overall Accuracy':<35} {metrics['accuracy']*100:>19.4f}%")
    print(f"{'Overall Error Rate':<35} {metrics['error_rate']*100:>19.4f}%")
    print(f"{'Cohen\\'s Kappa Score':<35} {metrics['kappa']:>20.4f}")
    print(f"{'Overall Precision':<35} {metrics['precision']*100:>19.4f}%")
    print(f"{'Overall Recall':<35} {metrics['recall']*100:>19.4f}%")
    print(f"{'Overall F1 Score':<35} {metrics['f1_score']*100:>19.4f}%")
    print(f"{'Overall Specificity':<35} {metrics['specificity']*100:>19.4f}%")
    print("="*80)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\nFinal GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

if __name__ == "__main__":
    main()